In [ ]:
import os
import struct
import numpy as np
import pandas as pd
import xgboost as xgb

# =========================
# Config
# =========================
CSV_PATH = "creditcard.csv"
OUT_DIR  = os.path.join("..", "xgbdata")  # ../xgbdata
N_TEST   = 32768
DEPTHS   = [8, 10, 12]
N_TREES  = 8
SEED     = 0

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Load
# =========================
df = pd.read_csv(CSV_PATH)
y = df["Class"].to_numpy().astype(np.float32, copy=False)
X = df.drop(columns=["Class"]).to_numpy().astype(np.float32, copy=False)

N, F = X.shape
rng = np.random.default_rng(SEED)

idx_all = np.arange(N)
idx_pos = idx_all[y == 1]
idx_neg = idx_all[y == 0]

pos_ratio = float(len(idx_pos)) / float(N)
n_pos_test = int(round(N_TEST * pos_ratio))
n_pos_test = min(n_pos_test, len(idx_pos))
n_neg_test = N_TEST - n_pos_test

pos_test = rng.choice(idx_pos, size=n_pos_test, replace=False)
neg_test = rng.choice(idx_neg, size=n_neg_test, replace=False)
idx_test = np.concatenate([pos_test, neg_test])
rng.shuffle(idx_test)

mask = np.ones(N, dtype=bool)
mask[idx_test] = False
idx_train = idx_all[mask]

Xtr, ytr = X[idx_train], y[idx_train]
Xte, yte = X[idx_test], y[idx_test]

# =========================
# Normalize X (train stats)
# =========================
x_mean = Xtr.mean(axis=0, keepdims=True)
x_std  = Xtr.std(axis=0, keepdims=True) + 1e-6
Xtr_n = (Xtr - x_mean) / x_std
Xte_n = (Xte - x_mean) / x_std

# imbalance helper
n_pos = float((ytr == 1).sum())
n_neg = float((ytr == 0).sum())
scale_pos_weight = (n_neg / max(n_pos, 1.0))

# =========================
# Save x_test.bin -> ../xgbdata/x_test.bin
# =========================
Xte_out = np.ascontiguousarray(Xte_n.astype(np.dtype("<f4"), copy=False))
N_out, F_out = Xte_out.shape

x_test_path = os.path.join(OUT_DIR, "x_test.bin")
with open(x_test_path, "wb") as f:
    f.write(struct.pack("<ii", N_out, F_out))
    f.write(Xte_out.tobytes(order="C"))

dtrain = xgb.DMatrix(Xtr_n, label=ytr, missing=np.nan)
dtest  = xgb.DMatrix(Xte_n, missing=np.nan)

# =========================
# Train params 
# =========================
base_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.05,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1.0,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "seed": SEED,
    "scale_pos_weight": scale_pos_weight,
    "tree_method": "hist",
}

for depth in DEPTHS:
    params = dict(base_params)
    params["max_depth"] = int(depth)

    bst = xgb.train(params, dtrain, num_boost_round=N_TREES)

    model_path = os.path.join(OUT_DIR, f"xgb_model_d{depth}.json")
    bst.save_model(model_path)

    pred = bst.predict(dtest, output_margin=True).astype(np.dtype("<f4"), copy=False)
    pred = np.ascontiguousarray(pred)

    pred_path = os.path.join(OUT_DIR, f"pred_d{depth}.bin")
    with open(pred_path, "wb") as f:
        f.write(struct.pack("<i", N_out))
        f.write(pred.tobytes(order="C"))


print("Done.")
print("Saved to:", OUT_DIR)
print("- x_test.bin")
for depth in DEPTHS:
    print(f"- xgb_model_d{depth}.json")
    print(f"- pred_d{depth}.bin")


Done.
Saved to: ../xgbdata
- x_test.bin
- xgb_model_d8.json
- pred_d8.bin
- xgb_model_d10.json
- pred_d10.bin
- xgb_model_d12.json
- pred_d12.bin


In [ ]:
import os
import json
import xgboost as xgb

def tree_stats_from_json(tree):
    def dfs(node, depth):
        if "leaf" in node:
            return depth, 1
        md = depth
        leaves = 0
        for ch in node.get("children", []):
            ch_md, ch_leaves = dfs(ch, depth + 1)
            if ch_md > md:
                md = ch_md
            leaves += ch_leaves
        return md, leaves
    return dfs(tree, 0)

def parse_base_score_value(v):
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, list) and len(v) > 0:
        return float(v[0])
    if isinstance(v, str):
        s = v.strip()
        if s.startswith("[") and s.endswith("]"):
            inner = s[1:-1].strip()
            if inner == "":
                return 0.0
            inner = inner.split(",")[0].strip()
            return float(inner)
        return float(s)
    return float(v)

def get_base_score(bst):
    cfg = json.loads(bst.save_config())
    v = cfg["learner"]["learner_model_param"]["base_score"]
    return parse_base_score_value(v)

def model_report(model_path):
    bst = xgb.Booster()
    bst.load_model(model_path)

    base_score = get_base_score(bst)

    dump = bst.get_dump(dump_format="json")
    per_tree = []
    for s in dump:
        tree = json.loads(s)
        md, leaves = tree_stats_from_json(tree)
        per_tree.append((md, leaves))

    model_max_depth = max(md for md, _ in per_tree) if per_tree else 0
    total_leaves = sum(leaves for _, leaves in per_tree)

    print(f"\n=== {os.path.basename(model_path)} ===")
    print(f"base_score={base_score}")
    for i, (md, leaves) in enumerate(per_tree):
        print(f"tree {i:02d}: max_depth={md}, leaves(paths)={leaves}")
    print(f"MODEL: max_depth={model_max_depth}, total_leaves={total_leaves}")

if __name__ == "__main__":
    paths = [
        "../xgbdata/xgb_model_d8.json",
        "../xgbdata/xgb_model_d10.json",
        "../xgbdata/xgb_model_d12.json",
    ]
    for p in paths:
        model_report(p)



=== xgb_model_d8.json ===
base_score=0.5
tree 00: max_depth=8, leaves(paths)=44
tree 01: max_depth=8, leaves(paths)=43
tree 02: max_depth=8, leaves(paths)=44
tree 03: max_depth=8, leaves(paths)=44
tree 04: max_depth=8, leaves(paths)=45
tree 05: max_depth=8, leaves(paths)=47
tree 06: max_depth=8, leaves(paths)=47
tree 07: max_depth=8, leaves(paths)=47
MODEL: max_depth=8, total_leaves=361

=== xgb_model_d10.json ===
base_score=0.5
tree 00: max_depth=10, leaves(paths)=55
tree 01: max_depth=10, leaves(paths)=56
tree 02: max_depth=10, leaves(paths)=56
tree 03: max_depth=10, leaves(paths)=56
tree 04: max_depth=10, leaves(paths)=59
tree 05: max_depth=10, leaves(paths)=61
tree 06: max_depth=10, leaves(paths)=62
tree 07: max_depth=10, leaves(paths)=64
MODEL: max_depth=10, total_leaves=469

=== xgb_model_d12.json ===
base_score=0.5
tree 00: max_depth=12, leaves(paths)=65
tree 01: max_depth=12, leaves(paths)=67
tree 02: max_depth=12, leaves(paths)=67
tree 03: max_depth=12, leaves(paths)=67
tree 